# Fig. 2E: organoid pseudotime APA enrichment (ORA ridgeplot; mouse-shared terms highlighted)

Ridgeplot (as in Fig. 1F and fig. S1J) of the organoid developmental APA program (inhibitory lineage, where the Reactome ORA is strongest; excitatory shares the RHO-GTPase terms). Each ridge = the per-gene LRT -log10(p) density for the term's members, coloured by the term's ORA padj. Curated to the terms shared with the mouse enrichment (RHO GTPases activate PKNs, WNT, telomere-stress senescence) plus organoid-neural terms (axon guidance, L1CAM). Unsigned -log10(chi-sq p) ranking, as for the mouse.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
REPO_ROOT <- Sys.getenv("M26_REPO_ROOT")
if (!nzchar(REPO_ROOT)) {
    REPO_ROOT <- normalizePath(getwd())
    while (!file.exists(file.path(REPO_ROOT, "environment", "apa-python.yml")) && REPO_ROOT != dirname(REPO_ROOT)) {
        REPO_ROOT <- dirname(REPO_ROOT)
    }
    Sys.setenv(M26_REPO_ROOT = REPO_ROOT)
}
DATA_ROOT <- Sys.getenv("M26_DATA_ROOT")
if (!nzchar(DATA_ROOT)) stop("Set M26_DATA_ROOT (see README)")
RESULTS_ROOT <- Sys.getenv("M26_RESULTS_ROOT", file.path(REPO_ROOT, "results"))

In [ ]:
source(paste0(REPO_ROOT, "/figures/style/paper_fig_style.R"))
suppressPackageStartupMessages({
    library(clusterProfiler)
    library(dplyr)
    library(stringr)
    library(ggridges)
    library(scales)
})
ORA_DIR <- paste0(DATA_ROOT, "/3utr/organoids/knoblich/utrcalling/ORA")
PANEL <- "fig2f_pseudotime_enrichment"
FIG <- "Fig2"
FIG_W <- HALF_W_IN
FIG_H <- mm_to_in(78) # 85 x 78 mm
LIN <- "inhibitory"

gsea <- readRDS(file.path(
    ORA_DIR,
    sprintf("gsea_reactome_%s_neglog10pval_chisq.rds", LIN)
))
ora <- readRDS(file.path(
    ORA_DIR,
    sprintf("ora_reactome_%s_neglog10pval_chisq.rds", LIN)
))

# per-gene -log10(LRT p) for every member of a term, + the term's ORA p.adjust for colour
build_long <- function(gsea_obj, ora_obj, descs) {
    gl <- gsea_obj@geneList
    gs <- gsea_obj@geneSets
    gres <- gsea_obj@result
    ores <- ora_obj@result
    out <- list()
    for (d in descs) {
        gid <- gres$ID[match(d, gres$Description)]
        if (is.na(gid) || is.null(gs[[gid]])) {
            message("skip: ", d)
            next
        }
        genes <- intersect(gs[[gid]], names(gl))
        padj <- ores$p.adjust[match(d, ores$Description)]
        out[[d]] <- data.frame(
            term = d,
            neglog10p = as.numeric(gl[genes]),
            ora_padj = padj,
            n = length(genes),
            stringsAsFactors = FALSE
        )
    }
    do.call(rbind, out)
}

In [ ]:
# curated: terms shared with the mouse enrichment (RHO GTPases, WNT, senescence) + organoid-neural terms
curated <- c(
    # Foreground the ORA-significant terms that are the developmental-wiring /
    # synaptic / cytoskeletal pathways recurrently disrupted in ASD & neurodevelopmental
    # disorders - a pathway-level link to disease (direct SFARI-gene overlap is not
    # significant).
    "Axon guidance",
    "Nervous system development",
    "L1CAM interactions",
    "Neurotransmitter receptors and postsynaptic signal transmission",
    "RHO GTPases activate PKNs",
    "RHO GTPases activate IQGAPs",
    "Signaling by WNT",
    "DNA Damage/Telomere Stress Induced Senescence"
)
df <- build_long(gsea, ora, curated)
lab <- setNames(
    sprintf(
        "%s  (n=%d)",
        str_wrap(curated, 26),
        sapply(curated, function(d) {
            m <- df$n[match(d, df$term)]
            ifelse(is.na(m), 0, m)
        })
    ),
    curated
)
df$label <- factor(lab[df$term], levels = rev(lab[curated]))
cat("genes per term:\n")
print(tapply(df$neglog10p, df$term, length)[curated])

In [ ]:
brks <- signif(
    exp(seq(log(min(df$ora_padj)), log(max(df$ora_padj)), length.out = 3)),
    1
)
p <- ggplot(df, aes(x = neglog10p, y = label, fill = ora_padj)) +
    geom_density_ridges(
        scale = 1.4,
        alpha = 0.92,
        color = "grey35",
        linewidth = 0.25,
        rel_min_height = 0.008,
        bandwidth = 0.5
    ) +
    scale_fill_gradientn(
        colours = rev(c("#ffffcc", "#fed976", "#fd8d3c", "#e31a1c", "#800026")),
        trans = "log10",
        limits = range(df$ora_padj),
        breaks = brks,
        labels = formatC(brks, format = "g", digits = 1),
        name = expression(ORA ~ italic(p)[adj]),
        guide = guide_colourbar(
            barwidth = 0.45,
            barheight = 3.2,
            ticks.colour = "grey30"
        )
    ) +
    geom_vline(
        xintercept = -log10(0.05),
        linetype = "dashed",
        color = "black",
        linewidth = 0.4
    ) +
    labs(
        x = expression(-log[10] ~ italic(p) * "-value  (per gene in set)"),
        y = NULL,
        title = "APA converges on\nneurodevelopmental-disease pathways",
        caption = "Inhibitory lineage (Reactome ORA); axon guidance / synaptic /\nRHO-GTPase pathways are disrupted in ASD & neurodevelopmental disorders"
    ) +
    coord_cartesian(xlim = c(0, 12)) +
    theme_paper +
    theme(
        axis.text.y = element_text(size = FONT_LEGEND, lineheight = 0.85),
        plot.title = element_text(size = FONT_ANNOT, hjust = 0.5),
        plot.caption = element_text(
            size = FONT_ANNOT - 1.5,
            hjust = 0.5,
            color = "grey30"
        ),
        legend.position = "right",
        panel.grid.major.x = element_line(linewidth = 0.25, color = "gray90"),
        panel.grid.major.y = element_blank()
    )
save_and_show(p, PANEL, FIG_W, FIG_H, figure = FIG)

## Half-width x quarter-height restyled ridgeplots + excitatory ORA (unassigned)

The main fig2f ridgeplot is HALF_W x 78 mm. Here: a half-width x quarter-height (85 x 58.75 mm)
restyle with no title and no x-axis caption, and the geneset origin ("Reactome") in a boxed annotation
top-right. Two panels: the inhibitory variant (`fig2f_pseudotime_enrichment__half_quarter`, Fig2 tree)
and an excitatory version (`fig2f_pseudotime_enrichment__exc_half_quarter` -> unassigned_panels).
The excitatory Reactome ORA is dominated by the RHO-GTPase / cytoskeletal axis (RHO GTPases activate
IQGAPs is the one strongly significant term; padj 9e-3), in line with the mouse Rac1/Cdc42 terms,
plus RAS/MAPK and senescence; the axon-guidance / synaptic neurodevelopmental terms are not
significant in excitatory (unlike inhibitory).


In [ ]:
# --- Half-width x quarter-height restyled ridgeplots: inhibitory (variant) + excitatory ---
# No title / no caption; geneset origin ("Reactome") in a boxed annotation top-right. Reuses
# build_long / ORA_DIR from above.
make_ridge_hq <- function(lin, curated, save, fig_dir) {
    gsea <- readRDS(file.path(
        ORA_DIR,
        sprintf("gsea_reactome_%s_neglog10pval_chisq.rds", lin)
    ))
    ora <- readRDS(file.path(
        ORA_DIR,
        sprintf("ora_reactome_%s_neglog10pval_chisq.rds", lin)
    ))
    df <- build_long(gsea, ora, curated)
    curated <- curated[curated %in% df$term]
    lab <- setNames(
        sprintf(
            "%s  (n=%d)",
            str_wrap(curated, 24),
            sapply(curated, function(d) {
                m <- df$n[match(d, df$term)]
                ifelse(is.na(m), 0, m)
            })
        ),
        curated
    )
    df$label <- factor(lab[df$term], levels = rev(lab[curated]))
    brks <- signif(
        exp(seq(log(min(df$ora_padj)), log(max(df$ora_padj)), length.out = 3)),
        1
    )
    p <- ggplot(df, aes(x = neglog10p, y = label, fill = ora_padj)) +
        geom_density_ridges(
            scale = 1.15,
            alpha = 0.92,
            color = "grey35",
            linewidth = 0.22,
            rel_min_height = 0.01,
            bandwidth = 0.5
        ) +
        scale_fill_gradientn(
            colours = rev(c(
                "#ffffcc",
                "#fed976",
                "#fd8d3c",
                "#e31a1c",
                "#800026"
            )),
            trans = "log10",
            limits = range(df$ora_padj),
            breaks = brks,
            labels = formatC(brks, format = "g", digits = 1),
            name = expression(ORA ~ italic(p)[adj]),
            guide = guide_colourbar(
                barwidth = 0.4,
                barheight = 2.6,
                ticks.colour = "grey30"
            )
        ) +
        geom_vline(
            xintercept = -log10(0.05),
            linetype = "dashed",
            color = "black",
            linewidth = 0.4
        ) +
        annotate(
            "label",
            x = Inf,
            y = Inf,
            label = "Reactome",
            hjust = 1.12,
            vjust = 1.5,
            size = 1.9,
            fill = "white",
            label.size = 0.25,
            label.padding = unit(0.4, "mm")
        ) +
        labs(
            x = expression(-log[10] ~ italic(p) * "  (per gene in set)"),
            y = NULL,
            title = NULL
        ) +
        # -log10(p) has a hard floor at 0: zero the left x-expansion + clip="on" so the KDE is cut
        # cleanly at 0 and no ridge/baseline bleeds outside the panel box (xmax bumped to 12.5).
        scale_x_continuous(
            expand = expansion(mult = c(0, 0.02)),
            breaks = seq(0, 12.5, 2.5)
        ) +
        scale_y_discrete(expand = expansion(mult = c(0.02, 0.20))) + # headroom for the top ridge
        coord_cartesian(xlim = c(0, 12.5), clip = "on") +
        theme_paper +
        theme(
            axis.text.y = element_text(
                size = FONT_LEGEND - 1,
                lineheight = 0.78
            ),
            axis.title.x = element_text(size = FONT_TICK),
            legend.position = "right",
            panel.border = element_rect(linewidth = 0.3, fill = NA), # thinner box
            panel.grid.major.x = element_line(
                linewidth = 0.22,
                color = "gray90"
            ),
            panel.grid.major.y = element_blank()
        )
    save_and_show(p, save, HALF_W_IN, QUARTER_H_IN, fig_dir = fig_dir)
}

UNASSIGNED <- file.path(OUT_ROOT, "Fig2", "unassigned_panels")

# inhibitory: same curated terms as the main fig2f, restyled at half x quarter (Fig2 tree)
make_ridge_hq(
    "inhibitory",
    c(
        "Axon guidance",
        "Nervous system development",
        "L1CAM interactions",
        "Neurotransmitter receptors and postsynaptic signal transmission",
        "RHO GTPases activate PKNs",
        "RHO GTPases activate IQGAPs",
        "Signaling by WNT",
        "DNA Damage/Telomere Stress Induced Senescence"
    ),
    "fig2f_pseudotime_enrichment__half_quarter",
    file.path(OUT_ROOT, "Fig2", "fig2f_pseudotime_enrichment__half_quarter")
)

# excitatory: RHO-GTPase-dominated convergence + RAS/MAPK + senescence -> unassigned_panels
make_ridge_hq(
    "excitatory",
    c(
        "RHO GTPases activate IQGAPs",
        "RHO GTPases activate PKNs",
        "RHO GTPase Effectors",
        "Signaling by Rho GTPases",
        "Oncogenic MAPK signaling",
        "Oxidative Stress Induced Senescence"
    ),
    "fig2f_pseudotime_enrichment__exc_half_quarter",
    UNASSIGNED
)

## chisq∩permutation intersection ORA: regenerated Reactome panels + combined Reactome/DisGeNET

The panels above use the LRT/chisq ORA foreground. Here we recompute the fills from the chisq ∩
permutation intersection ORA (canonical rds in `ORA/chisq_perm_intersection/`; genes significant
under both the parametric LRT and the non-parametric permutation null, the more conservative set).
Per-gene x-axis stays -log10(p_chisq); only the term ORA p_adj (fill) changes.

- `fig2f_pseudotime_enrichment__half_quarter` (inhibitory) regenerated: 7 terms - all still
  significant under the intersection; DNA Damage/Telomere Stress Induced Senescence dropped
  (p_adj 0.022 -> 0.066, NS).
- `fig2f_pseudotime_enrichment__exc_half_quarter` (excitatory) regenerated: the intersection
  promotes the RHO-GTPase axis to significance (RHO PKNs/Effectors/Signaling by Rho GTPases cross
  <0.05); Oxidative Stress Induced Senescence dropped (0.19, NS). 5 significant terms.
- `fig2f_pseudotime_enrichment__reactome_disgenet` (inhibitory): both databases faceted
  and strip-labelled as in fig1g. Under the intersection, DisGeNET shows a
  neurodevelopmental-disease block (Rett syndrome, congenital brain anomaly, epileptic seizures,
  infantile spasm, microlissencephaly). Curated to fit quarter height (one of the two RHO-GTPase
  Reactome terms and WNT dropped).


In [ ]:
# --- Intersection (chisq∩perm) ORA: regenerated Reactome panels + combined Reactome/DisGeNET ---
INT_DIR <- file.path(ORA_DIR, "chisq_perm_intersection")

# score = chisq gsea geneList; term genes + p_adj = intersection ORA (@geneSets / @result)
build_long2 <- function(geneList, ora, descs) {
    res <- ora@result
    gs <- ora@geneSets
    out <- list()
    for (d in descs) {
        id <- res$ID[match(d, res$Description)]
        if (is.na(id) || is.null(gs[[id]])) {
            message("skip: ", d)
            next
        }
        genes <- intersect(gs[[id]], names(geneList))
        out[[d]] <- data.frame(
            term = d,
            neglog10p = as.numeric(geneList[genes]),
            ora_padj = res$p.adjust[match(d, res$Description)],
            n = length(genes)
        )
    }
    do.call(rbind, out)
}
.gl <- function(db, lin) {
    readRDS(file.path(
        ORA_DIR,
        sprintf("gsea_%s_%s_neglog10pval_chisq.rds", db, lin)
    ))@geneList
}
.int <- function(db, lin) {
    readRDS(file.path(
        INT_DIR,
        sprintf("ora_%s_%s_neglog10pval_chisqperm.rds", db, lin)
    ))
}

build_df_int <- function(specs, lin) {
    df <- do.call(
        rbind,
        lapply(specs, function(s) {
            d <- build_long2(.gl(s$db, lin), .int(s$db, lin), s$descs)
            if (is.null(d)) {
                return(NULL)
            }
            d$database <- s$label
            d
        })
    )
    ord <- df %>%
        distinct(term, database, ora_padj) %>%
        arrange(
            factor(database, levels = sapply(specs, `[[`, "label")),
            ora_padj
        )
    ord$label <- sprintf(
        "%s  (n=%d)",
        str_wrap(ord$term, 24),
        sapply(ord$term, function(t) df$n[match(t, df$term)])
    )
    df$label <- factor(
        ord$label[match(df$term, ord$term)],
        levels = rev(ord$label)
    )
    df$database <- factor(df$database, levels = sapply(specs, `[[`, "label"))
    df
}

render_ridge_int <- function(
    df,
    save,
    fig_dir,
    facet = FALSE,
    ridge_scale = 1.15
) {
    brks <- signif(
        exp(seq(log(min(df$ora_padj)), log(max(df$ora_padj)), length.out = 3)),
        1
    )
    p <- ggplot(df, aes(x = neglog10p, y = label, fill = ora_padj)) +
        geom_density_ridges(
            scale = ridge_scale,
            alpha = 0.92,
            color = "grey35",
            linewidth = 0.22,
            rel_min_height = 0.01,
            bandwidth = 0.5
        ) +
        scale_fill_gradientn(
            colours = rev(c(
                "#ffffcc",
                "#fed976",
                "#fd8d3c",
                "#e31a1c",
                "#800026"
            )),
            trans = "log10",
            limits = range(df$ora_padj),
            breaks = brks,
            labels = formatC(brks, format = "g", digits = 1),
            name = expression(ORA ~ italic(p)[adj]),
            guide = guide_colourbar(
                barwidth = 0.4,
                barheight = 2.6,
                ticks.colour = "grey30"
            )
        ) +
        geom_vline(
            xintercept = -log10(0.05),
            linetype = "dashed",
            color = "black",
            linewidth = 0.4
        ) +
        labs(
            x = expression(-log[10] ~ italic(p) * "  (per gene in set)"),
            y = NULL,
            title = NULL
        ) +
        scale_x_continuous(
            expand = expansion(mult = c(0, 0.02)),
            breaks = seq(0, 12.5, 2.5)
        ) +
        scale_y_discrete(expand = expansion(mult = c(0.02, 0.22))) +
        coord_cartesian(xlim = c(0, 12.5), clip = "on") +
        theme_paper +
        theme(
            axis.text.y = element_text(
                size = FONT_LEGEND - 1,
                lineheight = 0.72
            ),
            axis.title.x = element_text(size = FONT_TICK),
            legend.position = "right",
            panel.border = element_rect(linewidth = 0.3, fill = NA),
            panel.grid.major.x = element_line(
                linewidth = 0.22,
                color = "gray90"
            ),
            panel.grid.major.y = element_blank()
        )
    if (facet) {
        p <- p +
            facet_grid(
                database ~ .,
                scales = "free_y",
                space = "free_y",
                switch = "y"
            ) +
            theme(
                strip.text.y.left = element_text(
                    size = FONT_TICK - 0.5,
                    face = "bold",
                    angle = 90
                ),
                strip.background = element_rect(
                    fill = "#F0F0F0",
                    linewidth = 0.4
                ),
                strip.placement = "outside",
                panel.spacing.y = unit(1.5, "pt")
            )
    }
    save_and_show(p, save, HALF_W_IN, QUARTER_H_IN, fig_dir = fig_dir)
}

UNASSIGNED <- file.path(OUT_ROOT, "Fig2", "unassigned_panels")

# inhibitory Reactome (7; drop DNA-damage senescence) -> overwrites half_quarter
render_ridge_int(
    build_df_int(
        list(list(
            db = "reactome",
            label = "Reactome",
            descs = c(
                "Axon guidance",
                "Nervous system development",
                "L1CAM interactions",
                "Neurotransmitter receptors and postsynaptic signal transmission",
                "RHO GTPases activate PKNs",
                "RHO GTPases activate IQGAPs",
                "Signaling by WNT"
            )
        )),
        "inhibitory"
    ),
    "fig2f_pseudotime_enrichment__half_quarter",
    file.path(OUT_ROOT, "Fig2", "fig2f_pseudotime_enrichment__half_quarter")
)

# excitatory Reactome (5; drop Oxidative Stress senescence) -> overwrites exc half_quarter
render_ridge_int(
    build_df_int(
        list(list(
            db = "reactome",
            label = "Reactome",
            descs = c(
                "RHO GTPases activate IQGAPs",
                "RHO GTPases activate PKNs",
                "RHO GTPase Effectors",
                "Signaling by Rho GTPases",
                "Oncogenic MAPK signaling"
            )
        )),
        "excitatory"
    ),
    "fig2f_pseudotime_enrichment__exc_half_quarter",
    UNASSIGNED
)

# combined Reactome(5) + DisGeNET(5) faceted, inhibitory (main-figure variant)
render_ridge_int(
    build_df_int(
        list(
            list(
                db = "reactome",
                label = "Reactome",
                descs = c(
                    "Axon guidance",
                    "Nervous system development",
                    "L1CAM interactions",
                    "Neurotransmitter receptors and postsynaptic signal transmission",
                    "RHO GTPases activate PKNs"
                )
            ),
            list(
                db = "dgn",
                label = "DisGeNET",
                descs = c(
                    "Rett Syndrome",
                    "Congenital anomaly of brain",
                    "Epileptic Seizures",
                    "Infantile Spasm",
                    "Microlissencephaly"
                )
            )
        ),
        "inhibitory"
    ),
    "fig2f_pseudotime_enrichment__reactome_disgenet",
    file.path(
        OUT_ROOT,
        "Fig2",
        "fig2f_pseudotime_enrichment__reactome_disgenet"
    ),
    facet = TRUE,
    ridge_scale = 1.0
)